# Day 2 — Streaming

## Core insight

**Streaming does NOT reduce total latency.** It optimizes 
*perceived* responsiveness by lowering Time-to-First-Token (TTFT).

Measured today (200-word output, Haiku):
- Sync: total 3.32s, user sees nothing until done
- Stream: TTFT 0.62s, total 3.16s
- Same total time, **5.4× faster perceived response**

TTFT advantage scales with output length. Short outputs (<1s sync) 
get near-zero benefit from streaming. Long outputs (>10s sync) 
get massive benefit (would be 30×+ for a 2000-word response).

## What streaming actually sends back

Not just text — a stream of typed events (Server-Sent Events / SSE).
The Anthropic SDK exposes two access levels:

- **High-level**: `stream.text_stream` — convenience iterator over 
  just text content. Most common use case.
- **Low-level**: `for event in stream` — full event objects with 
  `event.type` ∈ {`message_start`, `content_block_start`, 
  `content_block_delta`, `content_block_stop`, `message_delta`, 
  `message_stop`}. Needed for tool_use streaming or custom handling.

Why event types exist: a single Claude response can mix multiple 
content blocks (text → tool_use → more text). Event-based design 
supports this complexity in one stream.

## When to use streaming

| Scenario | Streaming? | Reason |
|---|---|---|
| Long natural-language response | ✅ | Big TTFT win, user reads as it streams |
| Short response (<50 tokens) | ❌ | Sync is already fast enough |
| JSON / structured output | ❌ | Half-rendered JSON is worse UX than waiting |
| Background batch job | ❌ | No human is watching; TTFT doesn't matter |
| Caller needs full response before acting | ❌ | Streaming adds complexity for no gain |

**NomNom application**:
- ❌ Food image recognition (outputs JSON) — sync
- ✅ Phase 5 meal recommendation (natural language) — stream
- ✅ Phase 5 fridge agent ("what should I cook?") — stream the text parts

## What I'll explore in this notebook

1. Hand-write `chat_streaming()` using `stream.text_stream`
2. Measure TTFT vs. sync on the same input
3. Explore low-level event types — see what Claude actually sends

## Implementation reference (high-level)

```python
def chat_streaming(model, max_tokens, messages):
    with client.messages.stream(
        model=model, max_tokens=max_tokens, messages=messages
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        final_message = stream.get_final_message()
    return final_message.content[0].text
```

Critical details:
- `.stream()` not `.create()`
- `with` context manager: streams are resources, must close cleanly
- `flush=True` in print: bypass Python stdout buffer for real-time display
- `get_final_message()`: retrieve full Message object (usage, stop_reason) 
  after stream ends — same shape as sync response

<Interview Q&A — Streaming>

1. Why need streaming? What does streaming send back? Common event types?

   Streaming optimizes Time-to-First-Token, not total latency. Total 
   generation time is unchanged; user sees first output ~5-50× sooner.

   It sends back a sequence of typed events (SSE):
   - `message_start`: stream opens, Message metadata
   - `content_block_start`: new output block begins (text or tool_use)
   - `content_block_delta`: incremental content streams here
   - `content_block_stop`: current block ends
   - `message_delta`: message-level state (usage, stop_reason)
   - `message_stop`: stream complete

   Multiple event types exist because one response can contain mixed 
   blocks (text + tool_use). Event-based design handles this in one stream.

2. How to implement streaming?

   Two patterns depending on what you need:

   **High-level (text only)** — what we used today:
```python
   def chat_streaming(model, max_tokens, messages):
       with client.messages.stream(
           model=model, max_tokens=max_tokens, messages=messages
       ) as stream:
           for text in stream.text_stream:
               print(text, end="", flush=True)
           final_message = stream.get_final_message()
       return final_message.content[0].text
```

   Key details:
   - `.stream()` instead of `.create()`
   - `with` context manager: streams are resources, must be properly closed
   - `flush=True` in print: bypass Python's stdout buffer for real-time display
   - `get_final_message()`: after stream ends, retrieve the full Message 
     object (with usage, stop_reason, etc.) — same shape as sync response

   **Low-level (full event control)** — for tool use, custom handling:
```python
   with client.messages.stream(...) as stream:
       for event in stream:
           if event.type == "content_block_delta":
               # incremental text or tool input
               pass
           elif event.type == "message_stop":
               # finalize
               pass
```

3. When applies / when not?

   **Yes**: long natural-language output, interactive UX, user actively 
   waiting.
   
   **No**: short output, JSON/structured output, batch job, caller 
   needs full response before acting.

   General rule: streaming is for "user reads natural language as it 
   streams", not for "machine consumes structured data".

</Interview Q&A — Streaming>

In [2]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()

In [36]:
# define 3 helper functions
def add_user_message(text, messages):
    user_message = {"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(text, messages):
    assistant_message = {"role":"assistant", "content":text}
    messages.append(assistant_message)

def chat(model, max_tokens, messages):
    response = client.messages.create(
        model = model,
        max_tokens=max_tokens,
        messages=messages
    )
    print("Input tokens:", response.usage.input_tokens)
    print("Output tokens:", response.usage.output_tokens)
    return response.content[0].text

In [24]:
model="claude-haiku-4-5-20251001"
max_tokens = 1000
messages = []

add_user_message(text="what is transformer, give me concise answer", messages=messages)

In [26]:
# Basic streaming implementation with create function 
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01F6FFukTs9PLgquUNqXx4u1', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Transformer\n\nA **transformer** is a deep learning architecture that processes sequential data (like text) using **', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDelta

In [33]:
# Simplified implementation with complet message
with client.messages.stream(
        model = model,
        max_tokens=max_tokens,
        messages=messages
) as stream:
    for txt in stream.text_stream:
        print(txt, end="*******************", flush=True)
    final_message = stream.get_final_message()
    print("Final message: ", final_message)
    

#******************* Transformer

A **transformer** is a deep learning architecture that processes sequential data (like text) using **attention******************* mechanisms** instead of recurrence.

## Key Features:

- **Self*******************-Attention**: Weighs the importance of each word relative to others, regardless******************* of distance
- **Parallel Processing**: Processes all tokens simultaneously (unlike RNNs), enabling faster******************* training
- **Positional Encoding**: Adds position information to capture word order

## Main Components:

1. **Encoder**: Transforms******************* input into meaningful representations
2. **Decoder**: Generates output from encoded representations
3. **Attention Layers**: Calculate relationships between all input******************* elements

## Real-World Examples:

- GPT, ChatGPT, Gem*******************ini (large language models)
- BERT (text understanding)
- Vision Transformers (image processing)

## Why It's Powerful*

In [28]:
print("Final message: ", final_message)

Final message:  ParsedMessage(id='msg_017nBu3JLC9TvoggRyFsSZST', container=None, content=[ParsedTextBlock(citations=None, text='# Transformer\n\nA **transformer** is a deep learning architecture that processes sequential data (like text) using **self-attention mechanisms** instead of recurrence.\n\n## Key features:\n- **Self-attention**: Weighs relationships between all input elements simultaneously\n- **Parallel processing**: Processes entire sequences at once (unlike RNNs)\n- **Scalability**: Works well with large datasets\n- **Position encoding**: Captures word order through positional embeddings\n\n## Common applications:\n- Language models (GPT, BERT)\n- Machine translation\n- Text generation\n- Vision tasks\n\n## Why it matters:\nTransformers enabled breakthrough progress in NLP and became the foundation for modern AI systems like ChatGPT, Claude, and Gemini.', type='text', parsed_output=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='

In [30]:
final_message.content[0].text

'# Transformer\n\nA **transformer** is a deep learning architecture that processes sequential data (like text) using **self-attention mechanisms** instead of recurrence.\n\n## Key features:\n- **Self-attention**: Weighs relationships between all input elements simultaneously\n- **Parallel processing**: Processes entire sequences at once (unlike RNNs)\n- **Scalability**: Works well with large datasets\n- **Position encoding**: Captures word order through positional embeddings\n\n## Common applications:\n- Language models (GPT, BERT)\n- Machine translation\n- Text generation\n- Vision tasks\n\n## Why it matters:\nTransformers enabled breakthrough progress in NLP and became the foundation for modern AI systems like ChatGPT, Claude, and Gemini.'

In [31]:
# define chat_streaming function helper
def chat_streaming(model, max_tokens, messages):
    with client.messages.stream(
        model=model,
        max_tokens=max_tokens,
        messages=messages
    ) as stream:
        for txt in stream.text_stream:
            pass
        final_message = stream.get_final_message()
    return final_message.content[0].text

In [32]:
chat_streaming(model, max_tokens, messages)

'# Transformer\n\nA **transformer** is a deep learning architecture that processes sequential data using **self-attention** mechanisms instead of recurrence.\n\n## Key Features:\n- **Self-Attention**: Weighs the importance of different positions in the input simultaneously\n- **Parallel Processing**: Can process all tokens at once (unlike RNNs)\n- **Scalability**: Works well with large datasets\n- **No Recurrence**: No hidden state passed between steps\n\n## Main Components:\n1. **Encoder**: Processes input\n2. **Decoder**: Generates output\n3. **Attention Layers**: Calculate relationships between all positions\n4. **Feed-Forward Networks**: Process each position separately\n\n## Applications:\n- NLP (BERT, GPT, T5)\n- Machine translation\n- Image recognition (Vision Transformer)\n- Speech processing\n\n## Advantages:\n✓ Faster training than RNNs  \n✓ Better for long sequences  \n✓ Highly parallelizable\n\n**Foundation of modern AI models** like ChatGPT and other large language models.

In [35]:
messages = []
add_user_message(
    text="Write a 200-word paragraph explaining what transformer architecture is, in plain English.",
    messages=messages
)

print("Streaming start:")
full_text = chat_streaming(model, max_tokens=500, messages=messages)
print()
print(f"\nFull text length: {len(full_text)} chars")

Streaming start:


Full text length: 1368 chars


In [37]:
import time

# === 同步 ===
messages_sync = []
add_user_message("Write a 200-word paragraph about transformer architecture.", messages_sync)

t0 = time.time()
sync_result = chat(model, 500, messages_sync)
t_sync_total = time.time() - t0
print(f"[Sync] Total time: {t_sync_total:.2f}s")
print(f"[Sync] User sees nothing for {t_sync_total:.2f}s, then full text at once")
print()

# === Streaming ===
messages_stream = []
add_user_message("Write a 200-word paragraph about transformer architecture.", messages_stream)

t0 = time.time()
first_chunk_time = None

with client.messages.stream(
    model=model, max_tokens=500, messages=messages_stream
) as stream:
    for txt in stream.text_stream:
        if first_chunk_time is None:
            first_chunk_time = time.time() - t0
        # 不 print,只是为了测时间纯粹一些
        pass
    stream.get_final_message()

t_stream_total = time.time() - t0
print(f"[Stream] Time to first chunk (TTFT): {first_chunk_time:.2f}s")
print(f"[Stream] Total time: {t_stream_total:.2f}s")
print()
print(f"Difference: TTFT is {t_sync_total/first_chunk_time:.1f}x faster than sync's perceived response")

Input tokens: 18
Output tokens: 267
[Sync] Total time: 3.32s
[Sync] User sees nothing for 3.32s, then full text at once

[Stream] Time to first chunk (TTFT): 0.62s
[Stream] Total time: 3.16s

Difference: TTFT is 5.4x faster than sync's perceived response
